# 07 — WEB-RAG Fact Checker — FRESH V3

**Marker:** `WEB_RAG_FACT_CHECKER_FRESH_V3`

This notebook replaces the old `fact_check_script()` workflow. It extracts
claims, retrieves a few web passages, verifies each claim against those
passages, and applies only evidence-grounded corrections.


## Install dependencies once

In [ ]:
# Uncomment and run once:
# %pip install -U ddgs trafilatura requests

## Load project

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.fact_checker import (
    build_checked_script_filename,
    build_fact_check_report,
    build_report_filename,
    extract_claims,
    find_edited_script_file,
    load_script,
    retrieve_evidence,
    rewrite_corrected_script,
    save_report,
    save_script,
    summarize_report,
    verify_claims,
)
from educational_shorts.prompts import load_prompt

print("WEB_RAG_FACT_CHECKER_FRESH_V3")
print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
EDITED_SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "edited_scripts"
CHECKED_SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "checked_scripts"
FACT_CHECKS_DIRECTORY = PROJECT_ROOT / "data" / "fact_checks"
RETRIEVAL_CACHE_DIRECTORY = PROJECT_ROOT / "data" / "retrieval_cache"

SCRIPT_FILENAME = "boston_tea_party_fact_check_test.json"

MAX_CLAIMS = 6
MAX_SEARCH_RESULTS = 8
MAX_SOURCES_PER_CLAIM = 3
MAX_EXCERPT_CHARS = 1000
CACHE_TTL_DAYS = 30
FORCE_REFRESH = False

TARGET_WORDS_PER_MINUTE = 145
MINIMUM_WORDS = 105
MAXIMUM_WORDS = 150

CLAIM_EXTRACTION_TEMPERATURE = 0.1
VERIFICATION_TEMPERATURE = 0.0
REWRITE_TEMPERATURE = 0.2
GENERATION_SEED = 42

In [ ]:
from educational_shorts.schemas import (
    ScriptSegment,
    VideoScript,
    VideoTopic,
)
from educational_shorts.fact_checker import (
    normalize_script,
    save_script,
)


history_test_script = VideoScript(
    topic=VideoTopic(
        title="What Really Caused the Boston Tea Party?",
        category_path=[
            "History",
            "American History",
            "American Revolution",
        ],
        learning_objective=(
            "Explain what happened at the Boston Tea Party and why its "
            "causes and consequences are often oversimplified."
        ),
    ),
    hook=ScriptSegment(
        segment_type="hook",
        narration=(
            "The Boston Tea Party was not just a group of angry colonists "
            "throwing tea into the harbor."
        ),
        estimated_seconds=1,
    ),
    sections=[
        ScriptSegment(
            segment_type="what happened",
            narration=(
                "On December 16, 1773, protesters boarded three ships in "
                "Boston and dumped 342 chests of East India Company tea "
                "into the water."
            ),
            estimated_seconds=1,
        ),
        ScriptSegment(
            segment_type="why they protested",
            narration=(
                "The Tea Act had sharply raised the price of tea, making "
                "it impossible for ordinary colonists to afford, so the "
                "protest was mainly about high prices."
            ),
            estimated_seconds=1,
        ),
        ScriptSegment(
            segment_type="the disguises",
            narration=(
                "Many participants dressed as Mohawk people mainly so "
                "British authorities could not identify and arrest them. "
                "They damaged only tea and left the ships largely intact."
            ),
            estimated_seconds=1,
        ),
        ScriptSegment(
            segment_type="historical impact",
            narration=(
                "The protest immediately began the American Revolution "
                "and united all thirteen colonies behind independence, "
                "making it the single decisive cause of the war."
            ),
            estimated_seconds=1,
        ),
    ],
    closing=ScriptSegment(
        segment_type="closing",
        narration=(
            "In one night, the Boston Tea Party transformed a simple tax "
            "dispute into a unified colonial war for freedom."
        ),
        estimated_seconds=1,
    ),
    full_narration="placeholder",
    word_count=1,
    estimated_total_seconds=1,
)

history_test_script = normalize_script(
    history_test_script,
    words_per_minute=TARGET_WORDS_PER_MINUTE,
)

history_test_path = (
    EDITED_SCRIPTS_DIRECTORY
    / "boston_tea_party_fact_check_test.json"
)

save_script(
    history_test_script,
    history_test_path,
)

print(f"Saved test script to: {history_test_path}")
print(f"Words: {history_test_script.word_count}")
print(f"Seconds: {history_test_script.estimated_total_seconds}")

## Load edited script

In [ ]:
script_path = find_edited_script_file(
    EDITED_SCRIPTS_DIRECTORY,
    SCRIPT_FILENAME,
)
edited_script = load_script(script_path)

print(f"Loaded: {script_path}")
print(f"Title: {edited_script.topic.title}")
print(f"Words: {edited_script.word_count}")

## Load prompt

In [ ]:
fact_checker_system_prompt = load_prompt("fact_checker")
print("Loaded retrieval-grounded prompt.")

## 07A — Extract claims

In [ ]:
claims = extract_claims(
    script=edited_script,
    system_prompt=fact_checker_system_prompt,
    max_claims=MAX_CLAIMS,
    temperature=CLAIM_EXTRACTION_TEMPERATURE,
    seed=GENERATION_SEED,
)

for claim in claims:
    print(f"{claim.claim_id} [{claim.importance.upper()}]")
    print(f"Segment: {claim.segment_type}")
    print(f"Claim: {claim.atomic_claim}")
    print(f"Query: {claim.search_query}")
    print()

## 07B — Retrieve evidence

In [ ]:
evidence_bundles = retrieve_evidence(
    claims=claims,
    cache_directory=RETRIEVAL_CACHE_DIRECTORY,
    max_search_results=MAX_SEARCH_RESULTS,
    max_sources_per_claim=MAX_SOURCES_PER_CLAIM,
    max_excerpt_chars=MAX_EXCERPT_CHARS,
    cache_ttl_days=CACHE_TTL_DAYS,
    force_refresh=FORCE_REFRESH,
)

## Inspect evidence

In [ ]:
for bundle in evidence_bundles:
    print("=" * 80)
    print(f"{bundle.claim.claim_id}: {bundle.claim.atomic_claim}")
    if bundle.retrieval_error:
        print(f"Note: {bundle.retrieval_error}")

    for source in bundle.sources:
        print()
        print(f"{source.title} — {source.domain}")
        print(source.url)
        print(source.excerpt)

## 07C — Verify claims

In [ ]:
verifications = verify_claims(
    evidence_bundles=evidence_bundles,
    system_prompt=fact_checker_system_prompt,
    temperature=VERIFICATION_TEMPERATURE,
    seed=GENERATION_SEED,
)

for item in verifications:
    print(
        f"{item.claim_id}: {item.verdict.upper()} "
        f"[{item.severity.upper()}]"
    )
    print(item.explanation)

    print(f"Evidence quote: {item.evidence_quote!r}")
    print(f"Evidence URL: {item.evidence_url}")

    if item.correction:
        print(f"Correction: {item.correction}")

    if item.supporting_urls:
        print("Supporting URLs:")
        for url in item.supporting_urls:
            print(f"  - {url}")

    print()

## 07D — Apply corrections

In [ ]:
corrected_script = rewrite_corrected_script(
    script=edited_script,
    evidence_bundles=evidence_bundles,
    verifications=verifications,
    system_prompt=fact_checker_system_prompt,
    target_wpm=TARGET_WORDS_PER_MINUTE,
    minimum_words=MINIMUM_WORDS,
    maximum_words=MAXIMUM_WORDS,
    temperature=REWRITE_TEMPERATURE,
    seed=GENERATION_SEED,
    max_attempts=3,
)

report = build_fact_check_report(
    claims=claims,
    evidence_bundles=evidence_bundles,
    verifications=verifications,
    corrected_script=corrected_script,
)

for name, value in summarize_report(report).items():
    print(f"{name}: {value}")

print(report.summary)

## Save outputs

In [ ]:
checked_script_path = (
    CHECKED_SCRIPTS_DIRECTORY
    / build_checked_script_filename(corrected_script)
)
report_path = (
    FACT_CHECKS_DIRECTORY
    / build_report_filename(corrected_script)
)

save_script(corrected_script, checked_script_path)
save_report(report, report_path)

print(f"Saved script: {checked_script_path}")
print(f"Saved report: {report_path}")

## Preview corrected script

In [ ]:
print(f"TITLE: {corrected_script.topic.title}\n")
print(f"HOOK ({corrected_script.hook.estimated_seconds}s)")
print(corrected_script.hook.narration)
print()

for index, section in enumerate(corrected_script.sections, start=1):
    print(
        f"{index}. {section.segment_type.upper()} "
        f"({section.estimated_seconds}s)"
    )
    print(section.narration)
    print()

print(f"CLOSING ({corrected_script.closing.estimated_seconds}s)")
print(corrected_script.closing.narration)
print()
print(f"WORD COUNT: {corrected_script.word_count}")
print(
    f"ESTIMATED TOTAL: "
    f"{corrected_script.estimated_total_seconds} seconds"
)